### middle ware


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## summerization middleware

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:qwen/qwen3.6-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [17]:
##run with thread id

config = {"configurable":{"thread_id":"test-1"}}

In [18]:
### alternative test data

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages:{response}")
    print(f"Messages:{len(response['messages'])}")
    

messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='6be76ee5-9963-415a-b612-9a132388c9e8'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what is 2+2?"\n2.  **Identify Core Question:** This is a basic arithmetic question asking for the sum of 2 and 2.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely. "2 + 2 equals 4."\n5.  **Check for Accuracy/Complexity:** The question is straightforward, no hidden tricks, no additional context needed. The answer is definitively 4.\n6.  **Output Generation:** Provide the direct answer.✅\n</think>\n\n2 + 2 equals 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 17, 'total_tokens': 175, 'completion_time': 0.303805246, 'completion_tokens_details': None, 'prompt_time': 0.001176694, 'prompt_tokens_details': None, 'queue_time': 0.0

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """search hotels- return long response to use more tokens."""
    return f"""Hotels in {city}:
    1. grand hotel - 5 star,$350/night,spa,pool,gym
    2.city inn - 4 star,$180/night,business center
    3.budget stay - 3 star,$75/night,free wifi"""


agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    tools  = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:qwen/qwen3.6-27b",
            trigger=("tokens",500),
            keep=("tokens",200)
        )
    ]
)
config = {"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4

In [4]:
# run test

cities = ["paris","london","tokyo","new york","dubai","singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content = f"find the hotels in {city}")]},
        config = config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}:~{tokens} tokens,{len(response["messages"])}messages")
    print(f"{(response['messages'])}")

paris:~166 tokens,4messages
[HumanMessage(content='find the hotels in paris', additional_kwargs={}, response_metadata={}, id='059d241c-1490-4556-99e8-81c988fbf887'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Identify the user\'s request**: The user wants to find hotels in Paris.\n2.  **Identify the available tool**: The `search_hotels` function is available.\n3.  **Check function parameters**: `search_hotels` requires a `city` parameter (type: string).\n4.  **Extract the parameter**: The city is "Paris".\n5.  **Construct the function call**: `search_hotels(city="Paris")`.\n6.  **Execute the function call**.\n7.  **Process the result**: Once the result is returned, I will present the hotels found.\n\n*Self-correction/Refinement*: The tool description says "return long response to use more tokens". This is just a hint to the model about the nature of the response, not an instruction to artificially expand it, but the response might be verbo

## Human in the loop

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [36]:
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [37]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [38]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject 'Hello'.


In [39]:
result


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='0aabf307-7c14-433c-9b37-3e4d2c9a273c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to send an email.\n2.  **Extract Parameters**:\n   - Recipient: `john@test.com`\n   - Subject: `Hello`\n   - Body: `How are you?`\n3.  **Match with Tool**: The `send_email_tool` matches the intent. It requires `recipient`, `subject`, and `body`.\n4.  **Construct Tool Call**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n5.  **Execute Tool Call**: Call `send_email_tool` with the extracted parameters.\n6.  **Formulate Response**: Acknowledge the action and confirm the email was sent. (Will happen after tool returns). I will just generate the tool call now.✅\n', 'tool_calls': [{'id': '5ra9h0aza', 'function': {'arguments': '

In [ ]:
#reject

